# Jaro-Winkler threshold for the OpenAlex author

`scripts/1_ground_truth/2_jw_match.py` gives each OpenAlex author that signs one of the researcher's papers

    score = JW(Semantic Scholar name, display_name) + JW(Semantic Scholar name, longest alternative name)   (0 to 2)

and the best author of each paper gets a vote only if `score >= MIN_SCORE`. Here the threshold is chosen from data, with labels that **do not use names**:

- **same person**: the only OpenAlex author who signs **all** the researcher's papers, for researchers with 5 or more (researchers where two or more authors sign them all are left out as ambiguous);
- **another person**: the coauthors of those researchers who sign **one** of their papers.

Sample: 8,000 researchers with `factuality_author = True` and papers in OpenAlex (`jw_signers_sample.parquet`: every author of their 10 most recent papers, with the score).

In [1]:
import importlib.util

import numpy as np
import pandas as pd

spec = importlib.util.spec_from_file_location("jw", "../../../scripts/1_ground_truth/2_jw_match.py")
import sys; sys.path.insert(0, "../../.."); jw = importlib.util.module_from_spec(spec); spec.loader.exec_module(jw)

signers = pd.read_parquet("../../../../results/summary/openalex_cascade/checks/jw_signers_sample.parquet")
n_papers = signers.groupby("researcher_id")["work_id"].nunique()
c = signers.groupby(["researcher_id", "author_id"]).agg(n=("work_id", "nunique"), score=("score", "max")).reset_index()
c["n_papers"] = c["researcher_id"].map(n_papers)

full = c[(c["n_papers"] >= 5) & (c["n"] == c["n_papers"])]
unambiguous = full.groupby("researcher_id").size().loc[lambda s: s == 1].index
same = full[full["researcher_id"].isin(unambiguous)]
other = c[c["researcher_id"].isin(unambiguous) & (c["n"] == 1)]
pd.Series({"researchers": len(unambiguous), "same-person pairs": len(same), "coauthor pairs": len(other)})

researchers           2166
same-person pairs     2166
coauthor pairs       78891
dtype: int64

## 1. Scores of the same person vs coauthors

In [2]:
pd.DataFrame({
    "same person": same["score"].quantile([0.01, 0.05, 0.1, 0.25, 0.5]),
    "coauthors": other["score"].quantile([0.01, 0.05, 0.1, 0.25, 0.5]),
}).round(3).rename_axis("quantile")

,same person,coauthors
quantile,,
0.01,0.973,0.723
0.05,1.323,0.819
0.10,1.455,0.872
0.25,1.586,0.957
0.50,1.738,1.052


In [3]:
rows = []
for t in np.round(np.arange(1.0, 1.95, 0.05), 2):
    tpr, fpr = (same["score"] >= t).mean(), (other["score"] >= t).mean()
    rows.append({"threshold": t, "same person passes %": 100 * tpr, "coauthors pass %": 100 * fpr,
                 "TPR − FPR": 100 * (tpr - fpr)})
by_score = pd.DataFrame(rows).round(2)
by_score

,threshold,same person passes %,coauthors pass %,TPR − FPR
0,1.00,98.75,64.72,34.03
1,1.05,98.38,50.50,47.88
2,1.10,97.60,36.54,61.06
3,1.15,96.95,24.67,72.28
4,1.20,96.72,15.20,81.52
5,1.25,96.26,8.78,87.48
6,1.30,95.29,4.97,90.32
7,1.35,94.60,2.73,91.86
8,1.40,93.12,1.57,91.55
9,1.45,90.86,1.06,89.80


## 2. The full rule: how often it picks the right person
One vote per paper for its best author (if over the threshold); from the most recent paper, the first author with 2 votes is the one (on this sample, only the first 10 papers are looked at).

In [4]:
truth = same.set_index("researcher_id")["author_id"]
sub = signers[signers["researcher_id"].isin(unambiguous)]
rows = []
for t in [1.2, 1.25, 1.3, 1.35, 1.4, 1.45, 1.5, 1.6, 1.7]:
    jw.MIN_SCORE = t
    chosen, _ = jw.choose(jw.votes(sub))
    chosen = chosen.set_index("researcher_id")["author_id"]
    right = (chosen == truth.reindex(chosen.index)).sum()
    rows.append({"threshold": t, "researchers with an author %": 100 * len(chosen) / len(truth),
                 "right person, of those chosen %": 100 * right / len(chosen),
                 "right person, of all %": 100 * right / len(truth)})
pd.DataFrame(rows).round(2)

,threshold,researchers with an author %,"right person, of those chosen %","right person, of all %"
0,1.20,99.86,94.91,94.78
1,1.25,99.86,94.87,94.74
2,1.30,99.54,94.67,94.23
3,1.35,98.85,94.72,93.63
4,1.40,97.55,94.46,92.15
5,1.45,95.80,94.12,90.17
6,1.50,91.87,93.57,85.96
7,1.60,79.13,91.83,72.67
8,1.70,62.47,91.43,57.11


## 3. Reading

- The score separates the person from their coauthors: the threshold that maximises TPR − FPR is the one to use (see table 1).
- With the full rule, the share of right picks among the chosen ones is about the same for any threshold up to ~1.45; above it, researchers are left without an author (they go to the cascade) without gaining accuracy.
- The ~5% of wrong picks do not depend on the threshold: on those, another author on the papers has a more similar name than the person.